# 06 · Transformación y carga en AWS RDS PostgreSQL

## Contexto
Fase final del pipeline ETL. Lee los datos desde S3, los transforma 
y los carga en la base de datos PostgreSQL en AWS RDS.

## Objetivo
- Leer datos históricos desde S3 (`extraccion_historica.json`)
- Transformar en 12 DataFrames estructurados
- Limpiar Foreign Keys inválidas antes de la carga
- Cargar en RDS PostgreSQL con soporte para INSERT histórico y UPSERT incremental

## Requisitos
- Credenciales AWS configuradas via IAM Role o `aws configure`
- Secrets Manager con credenciales RDS bajo el nombre `Postgre`
- Bucket S3: `project-api-load-rawg-cris`
- Esquema `rawg` creado en RDS (ver notebook `04`)

## Lógica de carga
- Archivo `extraccion_historica` → modo `INSERT` (carga inicial)
- Cualquier otro archivo → modo `UPSERT` (carga incremental)

## Output
Base de datos RDS PostgreSQL poblada con ~20.000 videojuegos

In [ ]:
# ============================================
# CONFIGURACIÓN DE PATHS E IMPORTS
# ============================================

import json
import pandas as pd
import boto3
from pprint import pprint
import psycopg2
from psycopg2 import sql
import sqlalchemy
from sqlalchemy import create_engine, text
from datetime import datetime
from io import StringIO
import sys
from pathlib import Path

# ──────────────────────────────────────────
# CONFIGURACIÓN DINÁMICA DE PATHS
# ──────────────────────────────────────────

# Detectar automáticamente la raíz del proyecto
ruta_actual = Path.cwd()
print(f"Directorio actual: {ruta_actual}")

# Subir niveles hasta encontrar la raíz del proyecto
proyecto_raiz = ruta_actual
while proyecto_raiz != proyecto_raiz.parent:
    if (proyecto_raiz / "bootstrap.py").exists():
        break
    proyecto_raiz = proyecto_raiz.parent

print(f"Raíz del proyecto: {proyecto_raiz}")

# Añadir rutas al sys.path
rutas_necesarias = [
    proyecto_raiz,                    # Raíz (para bootstrap)
    proyecto_raiz / "01_etl",         # Scripts ETL
    proyecto_raiz / "utils"           # Utilidades
]

for ruta in rutas_necesarias:
    ruta_str = str(ruta)
    if ruta_str not in sys.path:
        sys.path.insert(0, ruta_str)
        print(f"Añadida al path: {ruta.name}")

# ──────────────────────────────────────────
# IMPORTS DEL PROYECTO
# ──────────────────────────────────────────

# Importar bootstrap
try:
    import bootstrap
    print("bootstrap.py importado")
except ImportError as e:
    print(f"No se pudo importar bootstrap: {e}")

# Importar función de transformación
try:
    from transform_rawg import transf_rawg_data
    print("transf_rawg_data importada")
except ImportError:
    # Fallback: importar manualmente si el nombre de carpeta da problemas
    import importlib.util
    ruta_transform = proyecto_raiz / "01_etl" / "transform_rawg.py"
    spec = importlib.util.spec_from_file_location("transform_rawg", ruta_transform)
    transform_module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(transform_module)
    transf_rawg_data = transform_module.transf_rawg_data
    print("transf_rawg_data importada (método alternativo)")


In [ ]:
# === LEER DATOS RAWG DEL S3 BUCKET ===
# ========================================
import json
import boto3
import os

print("Leyendo datos históricos de la RAW...\n")

session = boto3.Session()
s3 = session.client(service_name = "s3")

from pprint import pprint
# Listar objetos del bucket
bucket_name = "project-api-load-rawg-cris"

response = s3.list_objects_v2(Bucket = bucket_name)

if "Contents" in response:
    for obj in response["Contents"]:
        print(obj["Key"], obj["Size"])
else:
    print("No hay objetos en el bucket.")


def leer_json_s3(bucket: str, key: str) -> dict | list:
    obj = s3.get_object(Bucket=bucket, Key=key)
    body = obj["Body"].read()
    datos = json.loads(body)
    return datos

# Leer el archivo 
key_archivo = "extraccion_historica.json"  
datos_raw = leer_json_s3(bucket_name, key_archivo)

print(f"Total de juegos leídos: {len(datos_raw)}")

In [ ]:
# === TRANSFORMAR ===
# ===================
import json
import transform_rawg

from transform_rawg import transf_rawg_data

print("Ejecutando transformación...")

dataframes = transf_rawg_data(datos, verbose = False)

# Extraer DataFrames
esrb_ratings = dataframes['esrb_ratings']
platforms = dataframes['platforms']
genres = dataframes['genres']
stores = dataframes['stores']
tags = dataframes['tags']
games = dataframes['games']
games_status = dataframes['games_status']
ratings_distribution = dataframes['ratings_distribution']
game_platforms = dataframes['game_platforms']
game_genres = dataframes['game_genres']
game_stores = dataframes['game_stores']
game_tags = dataframes['game_tags']

print("DataFrames listos para cargar\n")

In [ ]:
# === CREAR ESQUEMA  Y TABLAS EN DB DE RDS (SINTÁXIS SQl) ===
# ==============================================================

# LA TABLA FUE CREADA MANUALMENTE EN RDS: rawg-db
# En caso de crearla desde el aquí, el código sería:

# from utils.aws_secrets import get_rds_credentials

# creds = get_rds_credentials("Postgre")   

# def crear_bd(endopoint, puerto):
#     try:
#         conn = psycopg2.connect(
#             user = creds["username"],
#             password = creds["password"],
#             host = creds["host"],
#             port = creds["port"],
#             dbname = creds["dbname"],
#             sslmode="require"
#         )
#         conn.autocommit = True
#         cur = conn.cursor()

#         # Crear base de datos
#         cur.execute("CREATE DATABASE rawg-db;")
#         print("Base de datos 'rawg-db' creada")

#         cur.close()
#         conn.close()
#      except Exception as e:
#         print(f"Error de conexión/creación: {e}")

from utils.aws_secrets import get_rds_credentials

print("Obteniendo credenciales desde Secrets Manager...")
creds = get_rds_credentials("Postgre")
print("Credenciales obtenidas")


# Definir esquema SQL
schema_sql = """
CREATE SCHEMA IF NOT EXISTS rawg;

-- =========================
-- DIMENSIONES / CATÁLOGOS
-- =========================

CREATE TABLE IF NOT EXISTS rawg.esrb_ratings (
  esrb_id   INT PRIMARY KEY,
  esrb_name VARCHAR(100) NOT NULL
);

CREATE TABLE IF NOT EXISTS rawg.platforms (
  platform_id   INT PRIMARY KEY,
  platform_name VARCHAR(200) NOT NULL
);

CREATE TABLE IF NOT EXISTS rawg.genres (
  genre_id          INT PRIMARY KEY,
  genre_name        VARCHAR(100) NOT NULL,
  genre_games_count BIGINT
);

CREATE TABLE IF NOT EXISTS rawg.stores (
  store_id   INT PRIMARY KEY,
  store_name VARCHAR(200) NOT NULL
);

CREATE TABLE IF NOT EXISTS rawg.tags (
  tag_id          BIGINT PRIMARY KEY,
  tag_name        VARCHAR(200) NOT NULL,
  tag_language    VARCHAR(50),
  tag_games_count BIGINT
);

-- =========================
-- HECHO PRINCIPAL
-- =========================

CREATE TABLE IF NOT EXISTS rawg.games (
  game_id           BIGINT PRIMARY KEY,
  game_name         VARCHAR(500) NOT NULL,
  tba               BOOLEAN,
  released_ym       CHAR(7),
  updated_ym        CHAR(7),

  game_rating        NUMERIC(4,2),
  ratings_count      BIGINT,
  game_added         BIGINT,
  playtime           INTEGER,
  suggestions_count  BIGINT,

  esrb_id           INT NULL,
  created_at        TIMESTAMP DEFAULT CURRENT_TIMESTAMP,

  CONSTRAINT fk_games_esrb
    FOREIGN KEY (esrb_id) REFERENCES rawg.esrb_ratings(esrb_id)
);

CREATE TABLE IF NOT EXISTS rawg.games_status (
  game_id BIGINT PRIMARY KEY
    REFERENCES rawg.games(game_id) ON DELETE CASCADE,
  yet     BIGINT NOT NULL DEFAULT 0,
  owned   BIGINT NOT NULL DEFAULT 0,
  beaten  BIGINT NOT NULL DEFAULT 0,
  toplay  BIGINT NOT NULL DEFAULT 0,
  dropped BIGINT NOT NULL DEFAULT 0,
  playing BIGINT NOT NULL DEFAULT 0
);

-- Distribución de ratings por juego (típicamente 4 filas por juego)
CREATE TABLE IF NOT EXISTS rawg.ratings_distribution (
  game_id        BIGINT NOT NULL,
  ratingd_id     INT NOT NULL,
  ratingd_title  VARCHAR(50),
  ratingd_count  BIGINT DEFAULT 0,
  ratingd_percent NUMERIC(6,2) DEFAULT 0.0,
  PRIMARY KEY (game_id, ratingd_id),
  FOREIGN KEY (game_id) REFERENCES rawg.games(game_id) ON DELETE CASCADE
);

-- =========================
-- RELACIONES N:M (BRIDGES)
-- =========================

CREATE TABLE IF NOT EXISTS rawg.game_platforms (
  game_id     BIGINT NOT NULL,
  platform_id INT NOT NULL,
  released_at VARCHAR(10),
  PRIMARY KEY (game_id, platform_id),
  FOREIGN KEY (game_id) REFERENCES rawg.games(game_id) ON DELETE CASCADE,
  FOREIGN KEY (platform_id) REFERENCES rawg.platforms(platform_id) ON DELETE CASCADE
);

CREATE TABLE IF NOT EXISTS rawg.game_genres (
  game_id  BIGINT NOT NULL,
  genre_id INT NOT NULL,
  PRIMARY KEY (game_id, genre_id),
  FOREIGN KEY (game_id) REFERENCES rawg.games(game_id) ON DELETE CASCADE,
  FOREIGN KEY (genre_id) REFERENCES rawg.genres(genre_id) ON DELETE CASCADE
);

CREATE TABLE IF NOT EXISTS rawg.game_stores (
  game_id  BIGINT NOT NULL,
  store_id INT NOT NULL,
  PRIMARY KEY (game_id, store_id),
  FOREIGN KEY (game_id) REFERENCES rawg.games(game_id) ON DELETE CASCADE,
  FOREIGN KEY (store_id) REFERENCES rawg.stores(store_id) ON DELETE CASCADE
);

CREATE TABLE IF NOT EXISTS rawg.game_tags (
  game_id BIGINT NOT NULL,
  tag_id  BIGINT NOT NULL,
  PRIMARY KEY (game_id, tag_id),
  FOREIGN KEY (game_id) REFERENCES rawg.games(game_id) ON DELETE CASCADE,
  FOREIGN KEY (tag_id) REFERENCES rawg.tags(tag_id) ON DELETE CASCADE
);

-- =========================
-- ÍNDICES
-- =========================
CREATE INDEX IF NOT EXISTS ix_games_released ON rawg.games(released_ym);
CREATE INDEX IF NOT EXISTS ix_games_esrb_id ON rawg.games(esrb_id);
CREATE INDEX IF NOT EXISTS ix_ratings_distribution_game ON rawg.ratings_distribution(game_id);
CREATE INDEX IF NOT EXISTS ix_game_platforms_platform ON rawg.game_platforms(platform_id);
CREATE INDEX IF NOT EXISTS ix_game_genres_genre ON rawg.game_genres(genre_id);
CREATE INDEX IF NOT EXISTS ix_game_stores_store ON rawg.game_stores(store_id);
CREATE INDEX IF NOT EXISTS ix_game_tags_tag ON rawg.game_tags(tag_id);
"""

def crear_conexion_y_tablas():
    # Crea conexión a RDS y ejecuta el esquema SQL
    conn = None
    try:
        print("Conectando a RDS PostgreSQL...")
        conn = psycopg2.connect(
            user = creds["username"],
            password = creds["password"],
            host = creds["host"],
            port = creds["port"],
            dbname = creds["dbname"],
            sslmode="require"
        )
        conn.autocommit = True
        cur = conn.cursor()

        print("Creando esquema y tablas...")
        cur.execute(schema_sql)
        
        print("Esquema y tablas creadas con éxito")

        cur.close()
        conn.close()  
        return conn
        
    except psycopg2.OperationalError as e:
        print(f"Error de conexión a RDS: {e}")
        return None
    except psycopg2.Error as e:
        print(f"Error de PostgreSQL: {e}")
        return None
    except Exception as e:
        print(f"Error inesperado: {e}")
        return None

# Ejecutar creación
conn = crear_conexion_y_tablas()

if conn:
    print("\nBase de datos lista para recibir datos")
else:
    print("\nRevisar configuración de RDS o credenciales")


In [ ]:
# ===============================
# FUNCIÓN DE CARGA CON UPSERT
# ===============================

from sqlalchemy import create_engine
from sqlalchemy.pool import NullPool

print("Creando engine SQLAlchemy...")
engine = create_engine(
    f"postgresql://{creds['username']}:{creds['password']}@"
    f"{creds['host']}:{creds['port']}/{creds['dbname']}",
    poolclass=NullPool,
    connect_args={"sslmode": "require"}
)
print("Engine creado")

# Configuración de tablas
TABLA_CONFIG = {
    'orden_carga': [
        'esrb_ratings', 'platforms', 'genres', 'stores', 'tags',
        'games', 'games_status', 'ratings_distribution',
        'game_platforms', 'game_genres', 'game_stores', 'game_tags'
    ],
    
    'primary_keys': {
        'esrb_ratings': ['esrb_id'],
        'platforms': ['platform_id'],
        'genres': ['genre_id'],
        'stores': ['store_id'],
        'tags': ['tag_id'],
        'games': ['game_id'],
        'games_status': ['game_id'],
        'ratings_distribution': ['game_id', 'ratingd_id'],
        'game_platforms': ['game_id', 'platform_id'],
        'game_genres': ['game_id', 'genre_id'],
        'game_stores': ['game_id', 'store_id'],
        'game_tags': ['game_id', 'tag_id']
    },
    
    'columnas': {
        'esrb_ratings': ['esrb_id', 'esrb_name'],
        'platforms': ['platform_id', 'platform_name'],
        'genres': ['genre_id', 'genre_name', 'genre_games_count'],
        'stores': ['store_id', 'store_name'],
        'tags': ['tag_id', 'tag_name', 'tag_language', 'tag_games_count'],
        'games': ['game_id', 'game_name', 'tba', 'released_ym', 'updated_ym',
                  'game_rating', 'ratings_count', 'game_added', 'playtime',
                  'suggestions_count', 'esrb_id'],
        'games_status': ['game_id', 'yet', 'owned', 'beaten', 'toplay', 'dropped', 'playing'],
        'ratings_distribution': ['game_id', 'ratingd_id', 'ratingd_title',
                                 'ratingd_count', 'ratingd_percent'],
        'game_platforms': ['game_id', 'platform_id', 'released_at'],
        'game_genres': ['game_id', 'genre_id'],
        'game_stores': ['game_id', 'store_id'],
        'game_tags': ['game_id', 'tag_id']
    }
}

def upsert_dataframe(df, tabla_nombre, modo='upsert'):
    """
    Carga un DataFrame a PostgreSQL con INSERT o UPSERT
    """
    if df.empty:
        print(f" {tabla_nombre}: DataFrame vacío, omitiendo")
        return
    
    # Obtener columnas esperadas
    columnas_esperadas = TABLA_CONFIG['columnas'][tabla_nombre]
    
    # Filtrar y reordenar DataFrame según columnas esperadas
    columnas_disponibles = [col for col in columnas_esperadas if col in df.columns]
    df_limpio = df[columnas_disponibles].copy()
    
    # Advertir si faltan columnas
    columnas_faltantes = set(columnas_esperadas) - set(columnas_disponibles)
    if columnas_faltantes:
        print(f" {tabla_nombre}: Columnas faltantes: {columnas_faltantes}")
    
    tabla_completa = f"rawg.{tabla_nombre}"
    pk_columns = TABLA_CONFIG['primary_keys'][tabla_nombre]
    temp_table = f"temp_{tabla_nombre}"
    
    try:
        with engine.begin() as conn:
            # Cargar a tabla temporal
            df_limpio.to_sql(
                temp_table,
                conn,
                if_exists='replace',
                index=False,
                schema='rawg'
            )
            
            registros_nuevos = len(df_limpio)
            columnas_str = ', '.join(columnas_disponibles)
            
            if modo == 'insert':
                # MODO HISTÓRICO
                sql = f"""
                INSERT INTO {tabla_completa} ({columnas_str})
                SELECT {columnas_str} FROM rawg.{temp_table}
                ON CONFLICT DO NOTHING;
                """
                conn.execute(text(sql))
                print(f"  {tabla_nombre}: {registros_nuevos} registros insertados")
                
            else:
                # MODO INCREMENTAL
                pk_constraint = ', '.join(pk_columns)
                columnas_update = [col for col in columnas_disponibles if col not in pk_columns]
                
                if columnas_update:  # Solo si hay columnas para actualizar
                    set_clause = ', '.join([f"{col} = EXCLUDED.{col}" for col in columnas_update])
                    sql = f"""
                    INSERT INTO {tabla_completa} ({columnas_str})
                    SELECT {columnas_str} FROM rawg.{temp_table}
                    ON CONFLICT ({pk_constraint})
                    DO UPDATE SET {set_clause};
                    """
                else:  # Solo PKs (ej: tablas puente sin datos extra)
                    sql = f"""
                    INSERT INTO {tabla_completa} ({columnas_str})
                    SELECT {columnas_str} FROM rawg.{temp_table}
                    ON CONFLICT DO NOTHING;
                    """
                
                conn.execute(text(sql))
                print(f"  {tabla_nombre}: {registros_nuevos} registros procesados")
            
            # Limpiar temporal
            conn.execute(text(f"DROP TABLE IF EXISTS rawg.{temp_table}"))
            
    except Exception as e:
        print(f" Error en {tabla_nombre}: {e}")
        raise

def cargar_todos_los_dataframes(dataframes_dict, modo='upsert'):
    """Carga todos los DataFrames en orden correcto"""
    modo_texto = "CARGA HISTÓRICA (INSERT)" if modo == 'insert' else "CARGA INCREMENTAL (UPSERT)"
    print(f"\n{modo_texto}")
    print("=" * 60)
    
    total_registros = sum(len(df) for df in dataframes_dict.values() if not df.empty)
    print(f"Total de registros a procesar: {total_registros:,}")
    print()
    
    for tabla in TABLA_CONFIG['orden_carga']:
        if tabla in dataframes_dict:
            df = dataframes_dict[tabla]
            upsert_dataframe(df, tabla, modo=modo)
        else:
            print(f"{tabla}: No encontrado en dataframes")
    
    print("\n" + "=" * 60)
    print("Carga completada exitosamente")

print("\nFunciones de carga definidas y listas")


In [ ]:
# ============================================
# LIMPIAR FOREIGN KEYS ANTES DE CARGAR
# ============================================

print("Limpiando Foreign Keys inválidas...\n")

# En games: Convertir esrb_id = 0 a NULL
if not games.empty and 'esrb_id' in games.columns:
    # Contar cuántos tienen 0
    count_zeros = (games['esrb_id'] == 0).sum()
    
    # Convertir 0 a None (NULL en PostgreSQL)
    games['esrb_id'] = games['esrb_id'].replace(0, None)
    
    print(f"games: {count_zeros} registros con esrb_id=0 convertidos a NULL")

# Verificar que no haya otros valores inválidos
if not games.empty and 'esrb_id' in games.columns:
    # Obtener IDs válidos de esrb_ratings
    esrb_validos = set(esrb_ratings['esrb_id'].values)
    
    # Encontrar IDs en games que NO están en esrb_ratings
    games_esrb_ids = games['esrb_id'].dropna().unique()
    invalidos = [id for id in games_esrb_ids if id not in esrb_validos]
    
    if invalidos:
        print(f"IDs de ESRB inválidos encontrados: {invalidos}")
        # Convertir inválidos a NULL también
        games.loc[games['esrb_id'].isin(invalidos), 'esrb_id'] = None
        print(f"{len(invalidos)} IDs inválidos convertidos a NULL")
    else:
        print(f"Todas las Foreign Keys son válidas")

print("\nForeign Keys limpiadas correctamente")

In [ ]:
# ============================================
# EJECUTAR CARGA DE DATOS
# ============================================

# Preparar diccionario de DataFrames
dataframes_dict = {
    'esrb_ratings': esrb_ratings,
    'platforms': platforms,
    'genres': genres,
    'stores': stores,
    'tags': tags,
    'games': games,
    'games_status': games_status,
    'ratings_distribution': ratings_distribution,
    'game_platforms': game_platforms,
    'game_genres': game_genres,
    'game_stores': game_stores,
    'game_tags': game_tags
}

# Determinar modo según nombre del archivo

archivo_cargado = key_archivo  

if "historica" in archivo_cargado or "historico" in archivo_cargado:
    modo_carga = 'insert'
else:
    modo_carga = 'upsert'

print(f"Archivo detectado: {archivo_cargado}")
print(f"Modo seleccionado: {modo_carga.upper()}")

# Ejecutar carga
cargar_todos_los_dataframes(dataframes_dict, modo=modo_carga)

# Cerrar conexión
if conn:
    conn.close()
    print("\nConexión cerrada")

print("\nPipeline completado")


In [ ]:
# ============================
# === CONSULTAR UNA TABLA ===
# ============================

from utils.aws_secrets import get_rds_credentials
import pandas as pd

creds = get_rds_credentials("Postgre")   


# def ejecutar_consultas_ejemplo():
#     """Ejecuta consultas de ejemplo"""

print("\n" + "=" * 60)
print("CONSULTAS DE UNA TABLA")
print("=" * 60)

conn2 = None
try:
    print("Conectando a RDS PostgreSQL...")
    conn2 = psycopg2.connect(
        user = creds["username"],
        password = creds["password"],
        host = creds["host"],
        port = creds["port"],
        dbname = creds["dbname"],
        sslmode="require"
    )
    cur2 = conn2.cursor()
    cur2.execute('SELECT * FROM rawg.games LIMIT 10;')
    data = cur2.fetchall()

    colnames = [desc[0] for desc in cur2.description]
    df_games = pd.DataFrame(data = data, columns =  colnames)
    print(df_games.head(10))

    cur2.close()
    conn2.close()
  
    
except psycopg2.OperationalError as e:
    print(f"Error de conexión a RDS: {e}")
except Exception as e:
    print(f"Error inesperado: {e}")



        

In [ ]:
# ============================
# === CONSULTAS DE EJEMPLO ===
# ============================

from utils.aws_secrets import get_rds_credentials

creds = get_rds_credentials("Postgre")   

def ejecutar_consultas_ejemplo():
    """Ejecuta consultas de ejemplo"""
    
    print("\n" + "=" * 60)
    print("CONSULTAS DE EJEMPLO")
    print("=" * 60)

    conn1 = None
    try:
        print("Conectando a RDS PostgreSQL...")
        conn1 = psycopg2.connect(
            user = creds["username"],
            password = creds["password"],
            host = creds["host"],
            port = creds["port"],
            dbname = creds["dbname"],
            sslmode="require"
        )
        conn1.autocommit = True
        cur1 = conn1.cursor()
        
        # Consulta 1: Top 10 juegos mejor valorados
        print(f"\nTop 10 juegos mejor valorados:")
        query1 = """
            SELECT game_name, game_rating, ratings_count
            FROM rawg.games
            WHERE game_rating > 0
            ORDER BY game_rating DESC, ratings_count DESC
            LIMIT 10;
        """
        df1 = pd.read_sql(query1, conn1)
        print(df1.to_string(index=False))
        
        # Consulta 2: Juegos por género
        print(f"\nCantidad de juegos por género:")
        query2 = """
            SELECT g.genre_name, COUNT(*) as cantidad
            FROM rawg.game_genres gg
            JOIN rawg.genres g ON gg.genre_id = g.genre_id
            GROUP BY g.genre_name
            ORDER BY cantidad DESC;
        """
        df2 = pd.read_sql(query2, conn1)
        print(df2.to_string(index=False))
        
        # Consulta 3: Juegos por plataforma
        print(f"\nTop 10 plataformas con más juegos:")
        query3 = """
            SELECT p.platform_name, COUNT(*) as cantidad
            FROM rawg.game_platforms gp
            JOIN rawg.platforms p ON gp.platform_id = p.platform_id
            GROUP BY p.platform_name
            ORDER BY cantidad DESC
            LIMIT 10;
        """
        df3 = pd.read_sql(query3, conn1)
        print(df3.to_string(index=False))
               
        cur1.close()
        conn1.close()  
        return conn
        
    except psycopg2.OperationalError as e:
        print(f"Error de conexión a RDS: {e}")
        return None
    except psycopg2.Error as e:
        print(f"Error de PostgreSQL: {e}")
        return None
    except Exception as e:
        print(f"Error inesperado: {e}")
        return None

ejecutar_consultas_ejemplo()
